In [1]:
import os, gc, re, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from pandas.api.types import union_categoricals
from scipy import sparse
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             classification_report, roc_curve, precision_recall_curve)
from sklearn.model_selection import train_test_split

try:
    from IPython.display import display
except ImportError:          # allows the code to run as a plain script too
    display = print

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.6f}")

import hashlib
from scipy import stats
from sklearn.ensemble import RandomForestClassifier

verdicts = []                    # collected PASS / WARN / FAIL lines, summarised at the end
def verdict(name, status, detail):
    verdicts.append((status, name, detail))
    print(f"[{status}] {name}: {detail}")

In [2]:
MODEL_NAME = "Diagnostics"
MODEL_SLUG = "diagnostics"
RANDOM_STATE = 42
TRAIN_FRACTION = 0.80            # chronological 80/20 split (first 80 % of time = train, last 20 % = test)
VALIDATION_FRACTION = 0.10       # carved from the END of the training period (never from test)
THRESHOLD = 0.50                 # initial decision threshold on the AML Risk Score (tunable later)
MIN_TEST_POSITIVES = 10          # if the chronological test set has fewer laundering rows -> documented fallback
TARGET_PRECISION = 0.20          # project goal: at least 1 in 5 flagged transactions is truly laundering
TARGET_RECALL = 0.80             # project goal: catch at least 80 of every 100 laundering transactions
THIN_DAY_ROWS = 10_000           # a calendar day with fewer rows than this is treated as a "thin tail" day
DEBUG_NROWS = None               # e.g. 300_000 to smoke-test the notebook quickly; None = all rows
MAX_TRAIN_NEGATIVES = 500_000



def find_project_root():
    """Walk upwards from the notebook's folder until we find the AML-GNN-Project root
    (the folder that contains data/processed). Works from notebooks/ or models/."""
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "data" / "processed").is_dir():
            return cand
    raise FileNotFoundError("Could not find a folder containing data/processed. "
                            "Open the notebook from inside AML-GNN-Project or set PROJECT_ROOT manually.")

PROJECT_ROOT = find_project_root()          # e.g. PROJECT_ROOT = Path(r"C:/.../AML-GNN-Project")
DATA_RAW       = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
MODELS_DIR     = PROJECT_ROOT / "models"
OUTPUTS_DIR    = PROJECT_ROOT / "outputs"
RESULTS_DIR    = PROJECT_ROOT / "results"
for d in (MODELS_DIR, OUTPUTS_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

PATH_TRANSACTIONS      = DATA_PROCESSED / "transactions.csv"
PATH_TX_FEATURES       = DATA_PROCESSED / "transaction_features.csv"
PATH_ACCOUNT_BEHAVIOUR = DATA_PROCESSED / "account_behaviour.csv"
PATH_TEMPORAL          = DATA_PROCESSED / "temporal_features.csv"
PATH_GRAPH             = DATA_PROCESSED / "graph_features.csv"
PATH_BANKS             = DATA_PROCESSED / "banks.csv"
PATH_ACCOUNTS          = DATA_PROCESSED / "accounts.csv"
PATH_PATTERNS          = DATA_RAW / "LI-Small_Patterns.txt"

print("Project root :", PROJECT_ROOT)
for p in (PATH_TRANSACTIONS, PATH_TX_FEATURES, PATH_ACCOUNT_BEHAVIOUR, PATH_TEMPORAL, PATH_GRAPH, PATH_BANKS, PATH_ACCOUNTS):
    print(f"  {'OK     ' if p.exists() else 'MISSING'} {p.relative_to(PROJECT_ROOT)}")

Project root : E:\AML-GNN-Project
  OK      data\processed\transactions.csv
  OK      data\processed\transaction_features.csv
  OK      data\processed\account_behaviour.csv
  OK      data\processed\temporal_features.csv
  OK      data\processed\graph_features.csv
  OK      data\processed\banks.csv
  OK      data\processed\accounts.csv


In [3]:
def fingerprint(path, id_col=None):
    h = hashlib.md5()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 22), b""):
            h.update(block)
    out = {"file": path.name, "size_MB": round(path.stat().st_size / 1e6, 2), "md5": h.hexdigest()}
    return out

fp_rows = [fingerprint(p) for p in (PATH_TRANSACTIONS, PATH_TX_FEATURES, PATH_ACCOUNT_BEHAVIOUR, PATH_TEMPORAL, PATH_GRAPH) if p.exists()]
display(pd.DataFrame(fp_rows))
print("Compare this table (esp. md5) with your teammates'. If md5 differs, you are NOT working on the same data.")

,file,size_MB,md5
0,transactions.csv,692.130000,2a6c2a8fb41723fc0527f395e21d06a3
1,transaction_features.csv,776.650000,11b6faf2de5191a080172ab2e27e9873
2,account_behaviour.csv,771.790000,350525aba1ebefccca56102545ff37d8
3,temporal_features.csv,488.050000,dbb98d0cbb60d765dc0f6159b4795c64
4,graph_features.csv,828.880000,ac0c78d9493fc0ccc2b7f373fe1538bb


Compare this table (esp. md5) with your teammates'. If md5 differs, you are NOT working on the same data.


In [4]:
t0 = time.time()

# Memory-efficient dtypes: ints as int32, account/currency/format columns as 'category'
# (they have few unique values compared with 6.9M rows), Timestamp as category (only ~26k unique minutes).
TX_DTYPES = {
    "Transaction ID": "int32", "From Bank": "int32", "To Bank": "int32",
    "Account": "category", "Account.1": "category",
    "Amount Received": "float64", "Receiving Currency": "category",
    "Amount Paid": "float64", "Payment Currency": "category",
    "Payment Format": "category", "Is Laundering": "int8",
    "Timestamp": "category",
}
tx = pd.read_csv(PATH_TRANSACTIONS, usecols=list(TX_DTYPES), dtype=TX_DTYPES, nrows=DEBUG_NROWS)
print(f"Transactions loaded: {tx.shape[0]:,} rows x {tx.shape[1]} columns  ({time.time()-t0:.1f}s)")


def parse_timestamp_column(col):
    """Parse the (categorical) Timestamp column quickly by parsing only the unique values.
    Returns float seconds since 1970 (NaN where unparsable)."""
    col = col.astype("category")
    cats = pd.Series(col.cat.categories.astype(str))
    for fmt in ["%d-%m-%Y %H:%M", "%Y/%m/%d %H:%M", "%Y-%m-%d %H:%M", "%d/%m/%Y %H:%M", "%m/%d/%Y %H:%M"]:
        parsed = pd.to_datetime(cats, format=fmt, errors="coerce")
        if parsed.notna().mean() > 0.999:
            print(f"Timestamp format detected: {fmt}")
            break
    else:
        raise ValueError("Could not detect the Timestamp format - please add it to the list above.")
    secs = (parsed - pd.Timestamp("1970-01-01")).dt.total_seconds().to_numpy()
    lookup = np.append(secs, np.nan)               # code -1 (missing) -> last element -> NaN
    return lookup[col.cat.codes.to_numpy()]

ts_abs = parse_timestamp_column(tx["Timestamp"])
bad_ts = np.isnan(ts_abs)
print(f"Rows with unparsable Timestamp: {bad_ts.sum():,}")
tx = tx.loc[~bad_ts].drop(columns=["Timestamp"]).reset_index(drop=True)
tx["ts_abs"] = ts_abs[~bad_ts].astype(np.int64)      # seconds since 1970
del ts_abs, bad_ts
gc.collect()
print("Date range:", pd.to_datetime(tx["ts_abs"].min(), unit="s"), "->", pd.to_datetime(tx["ts_abs"].max(), unit="s"))

Transactions loaded: 6,924,049 rows x 12 columns  (16.0s)
Timestamp format detected: %Y/%m/%d %H:%M
Rows with unparsable Timestamp: 0
Date range: 2022-09-01 00:00:00 -> 2022-09-17 15:28:00


In [5]:
# transaction_features.csv: we only need the geographic columns from it. The other columns are either
# re-derivable from transactions.csv, or (Transaction Hour / Day of Week) are blank in the rows we inspected,
# so we derive hour / weekday ourselves from the Timestamp.
TF_USECOLS = ["Transaction ID", "Sender Bank Country", "Receiver Bank Country", "Is Cross Border"]
tf = pd.read_csv(PATH_TX_FEATURES, usecols=TF_USECOLS, nrows=DEBUG_NROWS,
                 dtype={"Transaction ID": "int32", "Sender Bank Country": "category",
                        "Receiver Bank Country": "category", "Is Cross Border": "category"})
print(f"Transaction features loaded: {tf.shape[0]:,} rows x {tf.shape[1]} columns")
print("Transaction ID unique in transaction_features:", tf["Transaction ID"].is_unique)

Transaction features loaded: 6,924,049 rows x 4 columns
Transaction ID unique in transaction_features: True


In [6]:
# =====================================================================================
# CHECK 1 - Is data/processed/transactions.csv a faithful copy of the raw file?
# `tx` is still in FILE ORDER here (we have not sorted yet), so row i can be compared with row i of the raw CSV.
# =====================================================================================
PATH_RAW_TRANS = DATA_RAW / "LI-Small_Trans.csv"
n_proc, pos_proc = len(tx), int(tx["Is Laundering"].sum())
print(f"Processed file: {n_proc:,} rows | {pos_proc:,} laundering | Transaction ID unique: {tx['Transaction ID'].is_unique} | "
      f"ID range {tx['Transaction ID'].min()}-{tx['Transaction ID'].max()} | contiguous: {tx['Transaction ID'].max() - tx['Transaction ID'].min() + 1 == n_proc}")
print("Expected for LI-Small (per your project notes): ~6.9 million rows and ~3,565 laundering cases.")
ok_counts = (DEBUG_NROWS is not None) or (6_800_000 <= n_proc <= 7_000_000 and 3_400 <= pos_proc <= 3_700)
verdict("Row / positive counts in expected range", "PASS" if ok_counts else "WARN", f"{n_proc:,} rows, {pos_proc:,} positives")

# --- is the processed file in chronological order? ---
dec = float(np.mean(np.diff(tx["ts_abs"].to_numpy()) < 0))
print(f"Share of consecutive rows whose timestamp goes BACKWARDS in the processed file: {dec:.2%} (0 % = already chronological)")

if PATH_RAW_TRANS.exists():
    raw = pd.read_csv(PATH_RAW_TRANS, usecols=["Timestamp", "From Bank", "To Bank", "Amount Paid", "Is Laundering"], nrows=DEBUG_NROWS,
                      dtype={"Timestamp": "category", "From Bank": "int32", "To Bank": "int32", "Amount Paid": "float64", "Is Laundering": "int8"})
    raw_ts = parse_timestamp_column(raw["Timestamp"])
    print(f"Raw file: {len(raw):,} rows | {int(raw['Is Laundering'].sum()):,} laundering | raw date range: "
          f"{pd.to_datetime(np.nanmin(raw_ts), unit='s')} -> {pd.to_datetime(np.nanmax(raw_ts), unit='s')}")
    m = min(len(raw), n_proc)
    same_ts = raw_ts[:m] == tx["ts_abs"].to_numpy()[:m]
    same_y  = raw["Is Laundering"].to_numpy()[:m] == tx["Is Laundering"].to_numpy()[:m]
    same_amt = np.abs(raw["Amount Paid"].to_numpy()[:m] - tx["Amount Paid"].to_numpy()[:m]) < 0.01
    same_bank = (raw["From Bank"].to_numpy()[:m] == tx["From Bank"].to_numpy()[:m]) & (raw["To Bank"].to_numpy()[:m] == tx["To Bank"].to_numpy()[:m])
    cmp_tbl = pd.DataFrame({"Column": ["Timestamp", "Is Laundering", "Amount Paid", "From/To Bank"],
                            "% rows identical to raw": [100 * same_ts.mean(), 100 * same_y.mean(), 100 * same_amt.mean(), 100 * same_bank.mean()]})
    display(cmp_tbl)
    verdict("Row count equals raw", "PASS" if len(raw) == n_proc else "FAIL", f"raw {len(raw):,} vs processed {n_proc:,}")
    verdict("Target column identical to raw", "PASS" if same_y.all() else "FAIL", f"{100 * same_y.mean():.4f}% identical")
    verdict("Timestamps identical to raw", "PASS" if same_ts.all() else "FAIL", f"{100 * same_ts.mean():.4f}% identical")
    if not same_ts.all():
        bad = np.where(~same_ts)[0]
        show = pd.DataFrame({"row": bad[:10], "raw": pd.to_datetime(raw_ts[bad[:10]], unit="s"), "processed": pd.to_datetime(tx["ts_abs"].to_numpy()[bad[:10]], unit="s")})
        print("First mismatching rows:"); display(show)
        # hypothesis: day and month were swapped by a spreadsheet
        pdt = pd.to_datetime(tx["ts_abs"].to_numpy()[bad], unit="s")
        swapped = pd.to_datetime(pd.DataFrame({"year": pdt.year, "month": pdt.day, "day": pdt.month, "hour": pdt.hour, "minute": pdt.minute}), errors="coerce")
        swapped_sec = (swapped - pd.Timestamp("1970-01-01")).dt.total_seconds().to_numpy()
        print(f"Mismatches explained by a DAY<->MONTH swap: {np.mean(swapped_sec == raw_ts[bad]):.1%}")
    print("Raw file: share of backwards timestamp steps:", f"{float(np.mean(np.diff(raw_ts) < 0)):.2%}")
    del raw, raw_ts
    gc.collect()
else:
    verdict("Cross-check against raw file", "WARN", f"{PATH_RAW_TRANS.name} not found in data/raw - cannot prove the processed file is faithful")

# --- which date format does the processed Timestamp really use? (dd-mm vs mm-dd) ---
tcol = pd.read_csv(PATH_TRANSACTIONS, usecols=["Timestamp"], dtype="category", nrows=DEBUG_NROWS)["Timestamp"]
counts = tcol.value_counts()
rows = []
for label, fmt in [("dd-mm-yyyy", "%d-%m-%Y %H:%M"), ("mm-dd-yyyy", "%m-%d-%Y %H:%M"), ("yyyy/mm/dd", "%Y/%m/%d %H:%M")]:
    parsed = pd.to_datetime(pd.Series(counts.index.astype(str)), format=fmt, errors="coerce")
    w = counts.to_numpy()
    ok = parsed.notna().to_numpy()
    months = parsed.dt.month.to_numpy()
    rows.append({"Interpretation": label, "% rows parsable": 100 * w[ok].sum() / w.sum(),
                 "Distinct months": len(set(months[ok])) if ok.any() else 0,
                 "Date span (days)": (parsed.max() - parsed.min()).days if ok.any() else np.nan})
display(pd.DataFrame(rows))
print("The correct interpretation parses ~100 % of rows into ONE month with a compact span (~17 days for LI-Small).")
del tcol, counts
gc.collect()

Processed file: 6,924,049 rows | 3,565 laundering | Transaction ID unique: True | ID range 1-6924049 | contiguous: True
Expected for LI-Small (per your project notes): ~6.9 million rows and ~3,565 laundering cases.
[PASS] Row / positive counts in expected range: 6,924,049 rows, 3,565 positives
Share of consecutive rows whose timestamp goes BACKWARDS in the processed file: 47.64% (0 % = already chronological)
Timestamp format detected: %Y/%m/%d %H:%M
Raw file: 6,924,049 rows | 3,565 laundering | raw date range: 2022-09-01 00:00:00 -> 2022-09-17 15:28:00


,Column,% rows identical to raw
0,Timestamp,100.000000
1,Is Laundering,100.000000
2,Amount Paid,100.000000
3,From/To Bank,100.000000


[PASS] Row count equals raw: raw 6,924,049 vs processed 6,924,049
[PASS] Target column identical to raw: 100.0000% identical
[PASS] Timestamps identical to raw: 100.0000% identical
Raw file: share of backwards timestamp steps: 47.64%


,Interpretation,% rows parsable,Distinct months,Date span (days)
0,dd-mm-yyyy,0.000000,0,NaN
1,mm-dd-yyyy,0.000000,0,NaN
2,yyyy/mm/dd,100.000000,1,16.000000


The correct interpretation parses ~100 % of rows into ONE month with a compact span (~17 days for LI-Small).


0

In [7]:
rows_before = len(tx)
tx = tx.merge(tf, on="Transaction ID", how="left", validate="one_to_one")   # 1 row per transaction, no duplication
assert len(tx) == rows_before, "Join changed the number of rows!"
print(f"Merged table: {tx.shape[0]:,} rows x {tx.shape[1]} columns  (rows before join: {rows_before:,})")
print("Transactions without a transaction_features row:", int(tx["Sender Bank Country"].isna().sum()))
del tf
gc.collect()

Merged table: 6,924,049 rows x 15 columns  (rows before join: 6,924,049)
Transactions without a transaction_features row: 0


0

In [8]:
# --- duplicates / validity ---
if not tx["Transaction ID"].is_unique:
    dup = tx["Transaction ID"].duplicated().sum()
    print(f"WARNING: {dup:,} duplicated Transaction IDs -> keeping first occurrence")
    tx = tx.drop_duplicates("Transaction ID", keep="first")
assert set(tx["Is Laundering"].unique()) <= {0, 1}, "Target must be binary 0/1"

for c in ["Amount Received", "Amount Paid"]:
    n_bad = int((tx[c].isna() | (tx[c] < 0)).sum())
    print(f"{c}: invalid (NaN/negative) values = {n_bad:,}")
    tx[c] = tx[c].fillna(0).clip(lower=0)

# Geographic columns: unknown -> explicit 'Unknown' level
for c in ["Sender Bank Country", "Receiver Bank Country"]:
    if "Unknown" not in tx[c].cat.categories:
        tx[c] = tx[c].cat.add_categories("Unknown")
    tx[c] = tx[c].fillna("Unknown")

# --- chronological ordering (stable sort, ties broken by Transaction ID) ---
tx = tx.sort_values(["ts_abs", "Transaction ID"], kind="stable").reset_index(drop=True)
print(f"\nSorted chronologically. Rows: {len(tx):,}")
print("First timestamp:", pd.to_datetime(tx['ts_abs'].iloc[0], unit='s'), "| Last timestamp:", pd.to_datetime(tx['ts_abs'].iloc[-1], unit='s'))

Amount Received: invalid (NaN/negative) values = 0
Amount Paid: invalid (NaN/negative) values = 0

Sorted chronologically. Rows: 6,924,049
First timestamp: 2022-09-01 00:00:00 | Last timestamp: 2022-09-17 15:28:00


In [9]:
n = len(tx)

# Account identity = (Bank, Account number) -> dense integer code. We never one-hot encode accounts.
u = union_categoricals([tx["Account"], tx["Account.1"]])
acct_num = u.codes.astype(np.int64)
ACCT_MULT = len(u.categories) + 1
src_key = tx["From Bank"].to_numpy(np.int64) * ACCT_MULT + acct_num[:n]
dst_key = tx["To Bank"].to_numpy(np.int64) * ACCT_MULT + acct_num[n:]
codes, uniques = pd.factorize(np.concatenate([src_key, dst_key]))
sender_code, receiver_code = codes[:n].astype(np.int64), codes[n:].astype(np.int64)
N_ACCOUNTS = len(uniques)
del u, acct_num, src_key, dst_key, codes, uniques
gc.collect()

from_bank = tx["From Bank"].to_numpy(np.int64)
to_bank   = tx["To Bank"].to_numpy(np.int64)

# Currency equality (needs a shared category vocabulary)
uc = union_categoricals([tx["Receiving Currency"], tx["Payment Currency"]])
is_currency_same = (uc.codes[:n] == uc.codes[n:]).astype(np.float32)
del uc

# Cross-border flag from transaction_features.csv (fallback: compare countries where the flag is missing)
def binary_from_category(s):
    s = s.astype("category")
    truthy = {"TRUE", "1", "1.0", "YES", "T", "Y"}
    lookup = np.array([1 if str(c).strip().upper() in truthy else 0 for c in s.cat.categories] + [0], dtype=np.float32)
    return lookup[s.cat.codes.to_numpy()]            # code -1 (missing) -> last element (0)

is_cross_border = binary_from_category(tx["Is Cross Border"])
cb_missing = tx["Is Cross Border"].isna().to_numpy()
if cb_missing.any():
    s_c = tx.loc[cb_missing, "Sender Bank Country"].astype(str).to_numpy()
    r_c = tx.loc[cb_missing, "Receiver Bank Country"].astype(str).to_numpy()
    is_cross_border[cb_missing] = ((s_c != r_c) & (s_c != "Unknown") & (r_c != "Unknown")).astype(np.float32)
print(f"Distinct accounts: {N_ACCOUNTS:,} | cross-border share: {is_cross_border.mean():.2%} | same-currency share: {is_currency_same.mean():.2%}")

# Plain numpy arrays used below
y         = tx["Is Laundering"].to_numpy(np.int8)
tx_ids    = tx["Transaction ID"].to_numpy(np.int64)
ts_abs    = tx["ts_abs"].to_numpy(np.int64)
t_rel     = ts_abs - ts_abs.min()                       # seconds since first transaction
amt_paid  = tx["Amount Paid"].to_numpy(np.float64)
amt_recv  = tx["Amount Received"].to_numpy(np.float64)
hour      = ((ts_abs // 3600) % 24).astype(np.float32)
dow       = (((ts_abs // 86400) + 3) % 7).astype(np.float32)   # 0 = Monday (1970-01-01 was a Thursday)

Distinct accounts: 705,907 | cross-border share: 10.81% | same-currency share: 98.57%


In [10]:
# =====================================================================================
# DATA-DISTRIBUTION CHECK: rows and laundering cases per calendar day
# A healthy chronological study needs a reasonably even stream of transactions. Days with very few rows
# (a "thin tail") but many laundering cases distort a time-based split and any time-trending feature.
# =====================================================================================
day_num = ts_abs // 86400
udays, day_inv, day_cnt = np.unique(day_num, return_inverse=True, return_counts=True)
day_inv = day_inv.ravel()
day_pos = np.bincount(day_inv, weights=y, minlength=len(udays)).astype(int)
daily = pd.DataFrame({"Date": pd.to_datetime(udays * 86400, unit="s").date, "Rows": day_cnt, "Laundering": day_pos})
daily["Laundering %"] = 100 * daily["Laundering"] / daily["Rows"]
daily["Thin day?"] = daily["Rows"] < THIN_DAY_ROWS
thin_day_flag = day_cnt < THIN_DAY_ROWS
is_thin_row = thin_day_flag[day_inv]                   # True for every transaction that falls on a thin day
with pd.option_context("display.float_format", lambda v: f"{v:,.3f}"):
    display(daily)
daily.to_csv(RESULTS_DIR / "data_daily_distribution.csv", index=False)
if thin_day_flag.any():
    print("!" * 90)
    print(f"WARNING: {int(thin_day_flag.sum())} day(s) have fewer than {THIN_DAY_ROWS:,} rows: {int(is_thin_row.sum()):,} rows in total, "
          f"holding {int(y[is_thin_row].sum()):,} of {int(y.sum()):,} laundering cases ({y[is_thin_row].mean():.1%} prevalence there vs {y[~is_thin_row].mean():.3%} elsewhere).")
    print("These look like the end of the data-generation window (pattern completions) rather than normal traffic.")
    print("Metrics are therefore also reported separately for the 'main window' and the 'thin tail'. Run notebook 10 to verify against the raw file.")
    print("!" * 90)

,Date,Rows,Laundering,Laundering %,Thin day?
0,2022-09-01,1524807,310,0.020,False
1,2022-09-02,1027758,350,0.034,False
2,2022-09-03,283326,292,0.103,False
3,2022-09-04,282476,291,0.103,False
4,2022-09-05,657397,370,0.056,False
5,2022-09-06,657170,380,0.058,False
6,2022-09-07,658504,368,0.056,False
7,2022-09-08,657938,402,0.061,False
8,2022-09-09,891573,367,0.041,False
9,2022-09-10,282877,301,0.106,False


!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
These look like the end of the data-generation window (pattern completions) rather than normal traffic.
Metrics are therefore also reported separately for the 'main window' and the 'thin tail'. Run notebook 10 to verify against the raw file.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


In [11]:
# What do the thin-tail rows look like? (tx still holds the raw columns here)
if is_thin_row.any():
    print(f"{int(is_thin_row.sum()):,} rows on thin days. Sample:")
    display(tx.loc[is_thin_row, ["Transaction ID", "From Bank", "To Bank", "Amount Paid", "Payment Format", "Is Laundering"]].head(15))
    first_seen = ~pd.Series(sender_code).duplicated().to_numpy()
    print(f"Sender accounts appearing for the FIRST time: thin-tail rows {first_seen[is_thin_row].mean():.1%} vs other rows {first_seen[~is_thin_row].mean():.2%}")
    print("Interpretation: if a tail is mostly laundering rows from accounts already active earlier, it is the tail end of laundering patterns that started inside the main window")
    print("(a property of the data generator, not a broken file). Confirm with CHECK 1 above: if timestamps match the raw file, the tail is genuine.")
    verdict("Thin-tail days present", "WARN", f"{int(is_thin_row.sum()):,} rows / {int(y[is_thin_row].sum()):,} laundering cases - evaluate main window and tail separately")
else:
    verdict("Thin-tail days present", "PASS", "no day has fewer than THIN_DAY_ROWS rows")

223 rows on thin days. Sample:


,Transaction ID,From Bank,To Bank,Amount Paid,Payment Format,Is Laundering
6923826,6857496,21566,22661,53.550000,ACH,0
6923827,6857497,21566,22661,"1,976.050000",ACH,1
6923828,4897730,25400,2514,"14,186.540000",ACH,1
6923829,5192118,2175,23041,"11,248.660000",ACH,1
6923830,4724745,1422,1422,"2,780.020000",ACH,0
6923831,4724746,1422,2293,"2,372.470000",ACH,1
6923832,5662041,21414,21414,"1,124,641.860000",ACH,0
6923833,5662042,21414,23726,"71,337.670000",ACH,0
6923834,5662043,21414,23726,"888,431.690000",ACH,1
6923835,6750833,349,349,"17,979.080000",ACH,0


Sender accounts appearing for the FIRST time: thin-tail rows 0.0% vs other rows 9.84%
Interpretation: if a tail is mostly laundering rows from accounts already active earlier, it is the tail end of laundering patterns that started inside the main window
(a property of the data generator, not a broken file). Confirm with CHECK 1 above: if timestamps match the raw file, the tail is genuine.
[WARN] Thin-tail days present: 223 rows / 134 laundering cases - evaluate main window and tail separately


In [12]:
# =====================================================================================
# CAUSAL ("point-in-time") FEATURES
# For every transaction we only use transactions that happened STRICTLY BEFORE it
# (same timestamp = excluded from the time windows). Nothing from the future and nothing from the
# target column is used. This replaces the full-dataset aggregates in account_behaviour.csv,
# temporal_features.csv and graph_features.csv (see the leakage audit below).
# =====================================================================================
WINDOWS = {"1 Hour": 3600, "24 Hours": 86400, "7 Days": 7 * 86400}
BIG = 2 ** 32                      # separates entities inside the combined (entity, time) search key


def causal_entity_features(ent, t, amt, new_pair_flag, windows=WINDOWS):
    """For each row, statistics about the PREVIOUS rows of the same entity (account).
    ent: int entity code per row | t: seconds | amt: amount | new_pair_flag: 1 if this (entity, counterparty)
    pair appears for the first time. Rows must be in chronological order. Fully vectorised."""
    n = len(ent)
    order = np.argsort(ent, kind="stable")                 # group by entity, keep chronological order inside group
    e, ts, a = ent[order], t[order], amt[order]
    nf = new_pair_flag[order].astype(np.float64)
    pos = np.arange(n)
    start = np.searchsorted(e, e, side="left")             # first row of this entity's block
    csum_a = np.concatenate([[0.0], np.cumsum(a)])
    csum_n = np.concatenate([[0.0], np.cumsum(nf)])
    key = e * BIG + ts
    hi = np.searchsorted(key, key, side="left")            # first row with the same (entity, time) -> excludes same-time rows
    prior_cnt = (pos - start).astype(np.float64)
    prior_amt = csum_a[pos] - csum_a[start]

    out = {
        "prior_count": prior_cnt,
        "prior_amount_total": prior_amt,
        "prior_amount_mean": np.divide(prior_amt, prior_cnt, out=np.zeros(n), where=prior_cnt > 0),
        "prior_unique_counterparties": csum_n[pos] - csum_n[start],   # distinct counterparties seen BEFORE this row
    }
    for name, w in windows.items():
        lo = np.searchsorted(key, e * BIG + np.maximum(ts - w, 0), side="left")
        out[f"count_{name}"] = (hi - lo).astype(np.float64)
        out[f"amount_{name}"] = csum_a[hi] - csum_a[lo]
    same = np.zeros(n, dtype=bool)
    same[1:] = e[1:] == e[:-1]
    out["minutes_since_prev"] = np.where(same, (ts - np.roll(ts, 1)) / 60.0, -1.0)   # -1 = no previous transaction

    result = {}
    for k, v in out.items():                                # undo the sorting
        arr = np.empty(n, dtype=np.float32)
        arr[order] = v
        result[k] = arr
    return result


def prior_counts(ent):
    """Number of earlier rows for the same entity (used for bank-level activity)."""
    order = np.argsort(ent, kind="stable")
    e = ent[order]
    pos = np.arange(len(e))
    res = np.empty(len(e), dtype=np.float32)
    res[order] = pos - np.searchsorted(e, e, side="left")
    return res


# "first time we see this (sender, receiver) pair"  (rows are in chronological order)
pair_key = sender_code * N_ACCOUNTS + receiver_code
is_new_pair = (~pd.Series(pair_key).duplicated(keep="first").to_numpy()).astype(np.float32)
del pair_key

t0 = time.time()
snd = causal_entity_features(sender_code,   t_rel, amt_paid, is_new_pair)     # outgoing history of the sender
rcv = causal_entity_features(receiver_code, t_rel, amt_recv, is_new_pair)     # incoming history of the receiver
print(f"Causal account features computed in {time.time()-t0:.1f}s")

Causal account features computed in 6.6s


In [13]:
f = {}
# ---- transaction-level ----
f["Amount Received"] = amt_recv.astype(np.float32)
f["Amount Paid"] = amt_paid.astype(np.float32)
f["Amount Paid / Received Ratio"] = np.divide(amt_paid, amt_recv, out=np.zeros(n), where=amt_recv > 0).astype(np.float32)
f["Is Currency Same"] = is_currency_same
f["Is Cross Border"] = is_cross_border
f["Is Self Transfer"] = (sender_code == receiver_code).astype(np.float32)
f["Is Same Bank"] = (from_bank == to_bank).astype(np.float32)
f["Transaction Hour"] = hour
f["Day of Week"] = dow
f["Is Weekend"] = (dow >= 5).astype(np.float32)
f["Is New Counterparty Pair"] = is_new_pair

# ---- sender (outgoing) and receiver (incoming) history - causal ----
def add_block(prefix, direction, res, counterpart):
    f[f"{prefix} Prior {direction} Txn Count"] = res["prior_count"]
    f[f"{prefix} Prior {direction} Amount Total"] = res["prior_amount_total"]
    f[f"{prefix} Prior {direction} Amount Mean"] = res["prior_amount_mean"]
    f[f"{prefix} Prior Unique {counterpart}"] = res["prior_unique_counterparties"]
    for w in WINDOWS:
        f[f"{prefix} {direction} Txns Last {w}"] = res[f"count_{w}"]
    f[f"{prefix} {direction} Amount Last 24 Hours"] = res["amount_24 Hours"]
    f[f"{prefix} Minutes Since Previous {direction} Txn"] = res["minutes_since_prev"]

add_block("Sender", "Outgoing", snd, "Receivers")
add_block("Receiver", "Incoming", rcv, "Senders")
f["Amount Paid / Sender Prior Mean"] = np.where(snd["prior_count"] > 0, amt_paid / (snd["prior_amount_mean"] + 1.0), 0.0).astype(np.float32)

# ---- bank-level activity (causal replacement for the full-period counts in banks.csv) ----
f["Sender Bank Prior Txn Count"] = prior_counts(from_bank)
f["Receiver Bank Prior Txn Count"] = prior_counts(to_bank)

feat = pd.DataFrame(f).astype(np.float32)
feat = feat.replace([np.inf, -np.inf], 0).fillna(0)
NUMERIC_COLS = list(feat.columns)

# Low-cardinality categorical features (encoded later, using TRAINING levels only)
CAT_COLS = ["Payment Format", "Receiving Currency", "Payment Currency", "Sender Bank Country", "Receiver Bank Country"]
cat_df = tx[CAT_COLS].copy()

print(f"Model-ready numeric feature table: {feat.shape[0]:,} rows x {feat.shape[1]} numeric columns")
print(f"Categorical columns (one-hot after the split): {CAT_COLS}")
assert len(feat) == len(y) == len(tx_ids)
assert not feat.isna().any().any()
display(feat.head(3).T)

del f, snd, rcv, is_new_pair
tx = tx[["Transaction ID"]]      # raw table no longer needed (everything lives in numpy arrays / feat / cat_df)
gc.collect()

Model-ready numeric feature table: 6,924,049 rows x 32 numeric columns
Categorical columns (one-hot after the split): ['Payment Format', 'Receiving Currency', 'Payment Currency', 'Sender Bank Country', 'Receiver Bank Country']


,0,1,2
Amount Received,"592,571.000000","2,941.560059",0.360000
Amount Paid,"592,571.000000","2,941.560059",0.360000
Amount Paid / Received Ratio,1.000000,1.000000,1.000000
Is Currency Same,1.000000,1.000000,1.000000
Is Cross Border,0.000000,0.000000,0.000000
Is Self Transfer,0.000000,1.000000,0.000000
Is Same Bank,0.000000,1.000000,0.000000
Transaction Hour,0.000000,0.000000,0.000000
Day of Week,3.000000,3.000000,3.000000
Is Weekend,0.000000,0.000000,0.000000


0

In [14]:
n = len(y)
cut = int(TRAIN_FRACTION * n)
train_idx = np.arange(0, cut)            # earliest 80 %  -> training (history)
test_idx  = np.arange(cut, n)            # latest 20 %    -> testing  (future). NOT shuffled.
SPLIT_MODE = "chronological"

if y[train_idx].sum() == 0 or y[test_idx].sum() < MIN_TEST_POSITIVES:
    print("!" * 90)
    print(f"WARNING: the chronological split is unusable (train positives = {int(y[train_idx].sum())}, "
          f"test positives = {int(y[test_idx].sum())}, minimum required in test = {MIN_TEST_POSITIVES}).")
    print("FALLBACK ACTIVATED: stratified RANDOM 80/20 split. Results below are NOT a true future-prediction test.")
    print("!" * 90)
    SPLIT_MODE = "stratified_random_FALLBACK"
    train_idx, test_idx = train_test_split(np.arange(n), test_size=1 - TRAIN_FRACTION, stratify=y, random_state=RANDOM_STATE)
    train_idx, test_idx = np.sort(train_idx), np.sort(test_idx)


def split_summary(name, idx):
    pos = int(y[idx].sum()); neg = len(idx) - pos
    return {"Set": name, "Rows": len(idx), "Normal": neg, "Laundering": pos, "Laundering %": 100 * pos / max(len(idx), 1)}

print(f"Split mode: {SPLIT_MODE}")
print(f"Train rows: {len(train_idx):,} | Test rows: {len(test_idx):,}")
print(f"Train laundering count: {int(y[train_idx].sum()):,} | Train normal count: {int((y[train_idx]==0).sum()):,}")
print(f"Test  laundering count: {int(y[test_idx].sum()):,} | Test  normal count: {int((y[test_idx]==0).sum()):,}")
print(f"Train laundering %: {100*y[train_idx].mean():.4f}% | Test laundering %: {100*y[test_idx].mean():.4f}%")
if SPLIT_MODE == "chronological":
    print("Train period:", pd.to_datetime(ts_abs[train_idx[0]], unit="s"), "->", pd.to_datetime(ts_abs[train_idx[-1]], unit="s"))
    print("Test  period:", pd.to_datetime(ts_abs[test_idx[0]], unit="s"), "->", pd.to_datetime(ts_abs[test_idx[-1]], unit="s"))
assert y[train_idx].sum() > 0 and y[test_idx].sum() > 0, "Both sets must contain laundering examples"

# ---- validation slice carved out of the TRAINING part only (used for early stopping / threshold study) ----
if SPLIT_MODE == "chronological":
    vcut = int((1 - VALIDATION_FRACTION) * len(train_idx))
    fit_pool, val_idx = train_idx[:vcut], train_idx[vcut:]           # validation = latest 10 % of the training period
else:
    fit_pool, val_idx = train_test_split(train_idx, test_size=VALIDATION_FRACTION, stratify=y[train_idx], random_state=RANDOM_STATE)
    fit_pool, val_idx = np.sort(fit_pool), np.sort(val_idx)
print(f"\nFit pool rows: {len(fit_pool):,} | Validation rows: {len(val_idx):,} (laundering in validation: {int(y[val_idx].sum()):,})")
VAL_HAS_BOTH = len(np.unique(y[val_idx])) == 2
if not VAL_HAS_BOTH:
    print("WARNING: validation slice has no laundering examples -> early stopping / threshold tuning on validation are disabled.")

# ---- training subset (documented, NOT silent): keep ALL positives, sample negatives; TEST is never sampled ----
rng = np.random.default_rng(RANDOM_STATE)
pos_ids = fit_pool[y[fit_pool] == 1]
neg_ids = fit_pool[y[fit_pool] == 0]
n_neg_available = len(neg_ids)
if MAX_TRAIN_NEGATIVES is not None and len(neg_ids) > MAX_TRAIN_NEGATIVES:
    neg_ids = np.sort(rng.choice(neg_ids, size=MAX_TRAIN_NEGATIVES, replace=False))
fit_idx = np.sort(np.concatenate([pos_ids, neg_ids]))

print("\n--- Training-subset strategy ---")
print(f"Positives kept : {len(pos_ids):,} of {len(pos_ids):,} (100%)")
print(f"Negatives kept : {len(neg_ids):,} of {n_neg_available:,} ({len(neg_ids)/max(n_neg_available,1):.1%})  [MAX_TRAIN_NEGATIVES={MAX_TRAIN_NEGATIVES}]")
print(f"ROWS ACTUALLY USED FOR TRAINING: {len(fit_idx):,}")
print(f"Test set kept at its natural class distribution: {len(test_idx):,} rows, {100*y[test_idx].mean():.4f}% laundering")

Split mode: chronological
Train rows: 5,539,239 | Test rows: 1,384,810
Train laundering count: 2,640 | Train normal count: 5,536,599
Test  laundering count: 925 | Test  normal count: 1,383,885
Train laundering %: 0.0477% | Test laundering %: 0.0668%
Train period: 2022-09-01 00:00:00 -> 2022-09-08 16:06:00
Test  period: 2022-09-08 16:06:00 -> 2022-09-17 15:28:00

Fit pool rows: 4,985,315 | Validation rows: 553,924 (laundering in validation: 326)

--- Training-subset strategy ---
Positives kept : 2,314 of 2,314 (100%)
Negatives kept : 500,000 of 4,983,001 (10.0%)  [MAX_TRAIN_NEGATIVES=500000]
ROWS ACTUALLY USED FOR TRAINING: 502,314
Test set kept at its natural class distribution: 1,384,810 rows, 0.0668% laundering


In [15]:
test_thin = is_thin_row[test_idx]
print(f"Test rows on thin-tail days: {int(test_thin.sum()):,} of {len(test_idx):,}")
print(f"Test laundering cases on thin-tail days: {int(y[test_idx][test_thin].sum()):,} of {int(y[test_idx].sum()):,} "
      f"({y[test_idx][test_thin].sum() / max(y[test_idx].sum(), 1):.1%})")
print(f"Train rows on thin-tail days: {int(is_thin_row[train_idx].sum()):,}")
if SPLIT_MODE == "chronological":
    print("Cut between train and test at:", pd.to_datetime(ts_abs[test_idx[0]], unit="s"))

Test rows on thin-tail days: 223 of 1,384,810
Test laundering cases on thin-tail days: 134 of 925 (14.5%)
Train rows on thin-tail days: 0
Cut between train and test at: 2022-09-08 16:06:00


In [16]:
# =====================================================================================
# CHECK 3 - LEAKAGE TESTS
# =====================================================================================
# 3a. strict time ordering
if SPLIT_MODE == "chronological":
    ok = ts_abs[train_idx].max() <= ts_abs[test_idx].min()
    verdict("3a Train period ends before test period starts", "PASS" if ok else "FAIL",
            f"latest train {pd.to_datetime(ts_abs[train_idx].max(), unit='s')} | earliest test {pd.to_datetime(ts_abs[test_idx].min(), unit='s')}")
else:
    verdict("3a Chronological split", "WARN", "fallback RANDOM split is active - not a future-prediction test")
verdict("3a No transaction in both train and test", "PASS" if len(np.intersect1d(tx_ids[train_idx], tx_ids[test_idx])) == 0 else "FAIL", "Transaction ID overlap check")

# 3b. FUTURE-INVARIANCE TEST: delete all rows after position m and recompute the causal features.
#     If any feature of an earlier row changes, that feature used future information.
m = int(0.6 * len(y))
pk = sender_code[:m] * N_ACCOUNTS + receiver_code[:m]
new_m = (~pd.Series(pk).duplicated(keep="first").to_numpy()).astype(np.float32)
snd_m = causal_entity_features(sender_code[:m], t_rel[:m], amt_paid[:m], new_m)
rcv_m = causal_entity_features(receiver_code[:m], t_rel[:m], amt_recv[:m], new_m)
pairs = {"Sender Prior Outgoing Txn Count": snd_m["prior_count"], "Sender Prior Outgoing Amount Total": snd_m["prior_amount_total"],
         "Sender Prior Unique Receivers": snd_m["prior_unique_counterparties"], "Sender Outgoing Txns Last 1 Hour": snd_m["count_1 Hour"],
         "Sender Outgoing Txns Last 24 Hours": snd_m["count_24 Hours"], "Sender Outgoing Txns Last 7 Days": snd_m["count_7 Days"],
         "Sender Outgoing Amount Last 24 Hours": snd_m["amount_24 Hours"], "Sender Minutes Since Previous Outgoing Txn": snd_m["minutes_since_prev"],
         "Receiver Prior Incoming Txn Count": rcv_m["prior_count"], "Receiver Incoming Txns Last 24 Hours": rcv_m["count_24 Hours"],
         "Receiver Prior Unique Senders": rcv_m["prior_unique_counterparties"],
         "Sender Bank Prior Txn Count": prior_counts(from_bank[:m]), "Receiver Bank Prior Txn Count": prior_counts(to_bank[:m])}
bad_cols = [c for c, v in pairs.items() if not np.allclose(feat[c].to_numpy()[:m], v, rtol=1e-4, atol=1e-3)]
verdict("3b Future-invariance of causal features", "PASS" if not bad_cols else "FAIL",
        "features of earlier rows are unchanged when later rows are deleted" if not bad_cols else f"these features depend on the future: {bad_cols}")
del pk, new_m, snd_m, rcv_m, pairs
gc.collect()

# 3c. single-feature strength: a single feature that separates the classes almost perfectly is a leak suspect
rng_chk = np.random.default_rng(RANDOM_STATE)
def sample_rows(idx, max_neg=200_000):
    pos_i, neg_i = idx[y[idx] == 1], idx[y[idx] == 0]
    if len(neg_i) > max_neg:
        neg_i = rng_chk.choice(neg_i, max_neg, replace=False)
    return np.concatenate([pos_i, neg_i])
rows_tr, rows_te = sample_rows(fit_idx), sample_rows(test_idx)
auc_rows = []
for col in NUMERIC_COLS:
    r = {"Feature": col}
    for nm, ids in (("Train AUC", rows_tr), ("Test AUC", rows_te)):
        a = roc_auc_score(y[ids], feat[col].to_numpy()[ids]); r[nm] = a
    r["Strength (test)"] = max(r["Test AUC"], 1 - r["Test AUC"])
    auc_rows.append(r)
auc_df = pd.DataFrame(auc_rows).sort_values("Strength (test)", ascending=False)
print("Single-feature ROC-AUC (0.5 = useless, 1.0 / 0.0 = perfect). Top 12:")
display(auc_df.head(12))
strongest = float(auc_df["Strength (test)"].iloc[0])
verdict("3c No single feature is near-perfect", "PASS" if strongest < 0.95 else "FAIL",
        f"strongest single feature: {auc_df['Feature'].iloc[0]} (AUC strength {strongest:.3f})")

# 3d. LABEL-SHUFFLE SANITY TEST: with randomly permuted labels no model should beat chance on the test set.
small_neg = fit_idx[y[fit_idx] == 0]
small_neg = rng_chk.choice(small_neg, min(150_000, len(small_neg)), replace=False)
fit_small = np.sort(np.concatenate([fit_idx[y[fit_idx] == 1], small_neg]))
Xtr, Xte = feat.iloc[fit_small].to_numpy(), feat.iloc[test_idx].to_numpy()
prevalence = float(y[test_idx].mean())
def quick_rf(labels):
    return RandomForestClassifier(n_estimators=80, max_depth=12, min_samples_leaf=20, class_weight="balanced_subsample",
                                  n_jobs=-1, random_state=RANDOM_STATE).fit(Xtr, labels)
rf_real = quick_rf(y[fit_small]); s_real = rf_real.predict_proba(Xte)[:, 1]
rf_shuf = quick_rf(rng_chk.permutation(y[fit_small])); s_shuf = rf_shuf.predict_proba(Xte)[:, 1]
res = pd.DataFrame({"Model": ["RF - real labels", "RF - SHUFFLED labels", "No-skill reference"],
                    "ROC-AUC": [roc_auc_score(y[test_idx], s_real), roc_auc_score(y[test_idx], s_shuf), 0.5],
                    "PR-AUC": [average_precision_score(y[test_idx], s_real), average_precision_score(y[test_idx], s_shuf), prevalence]})
display(res)
shuf_ok = res.loc[1, "ROC-AUC"] < 0.56 and res.loc[1, "PR-AUC"] < 3 * prevalence + 0.002
verdict("3d Shuffled-label model is at chance level", "PASS" if shuf_ok else "FAIL",
        f"ROC-AUC {res.loc[1, 'ROC-AUC']:.3f}, PR-AUC {res.loc[1, 'PR-AUC']:.5f} (prevalence {prevalence:.5f})")

# 3e. TRAIN -> TEST DISTRIBUTION SHIFT (not a leak, but a classic cause of 'works in validation, fails on test')
samp_tr = rng_chk.choice(fit_idx, min(50_000, len(fit_idx)), replace=False); samp_te = rng_chk.choice(test_idx, min(50_000, len(test_idx)), replace=False)
ks_rows = []
for col in NUMERIC_COLS:
    a, b = feat[col].to_numpy()[samp_tr], feat[col].to_numpy()[samp_te]
    ks_rows.append({"Feature": col, "KS statistic": stats.ks_2samp(a, b).statistic, "Median train": np.median(a), "Median test": np.median(b)})
ks_df = pd.DataFrame(ks_rows).sort_values("KS statistic", ascending=False)
print("Features whose distribution differs most between training and test (KS statistic, 0 = same, 1 = disjoint):")
display(ks_df.head(10))
drift_cols = list(ks_df.loc[ks_df["KS statistic"] > 0.30, "Feature"])
verdict("3e Train/test feature drift", "PASS" if not drift_cols else "WARN",
        "no strongly drifting feature" if not drift_cols else f"{len(drift_cols)} feature(s) with KS > 0.30, e.g. {drift_cols[:4]} (cumulative counters grow with time)")

# 3f. ABLATION: does dropping the drifting (time-trending) features change the result?
if drift_cols:
    keep = [c for c in NUMERIC_COLS if c not in drift_cols]
    Xtr2, Xte2 = feat[keep].iloc[fit_small].to_numpy(), feat[keep].iloc[test_idx].to_numpy()
    rf_nodrift = RandomForestClassifier(n_estimators=80, max_depth=12, min_samples_leaf=20, class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE).fit(Xtr2, y[fit_small])
    s_nd = rf_nodrift.predict_proba(Xte2)[:, 1]
    abl = pd.DataFrame({"Feature set": ["all features", f"without {len(drift_cols)} drifting features"],
                        "ROC-AUC": [roc_auc_score(y[test_idx], s_real), roc_auc_score(y[test_idx], s_nd)],
                        "PR-AUC": [average_precision_score(y[test_idx], s_real), average_precision_score(y[test_idx], s_nd)]})
    display(abl)
    print("If PR-AUC is similar or better WITHOUT them, remove / rate-normalise those features in notebooks 11-14.")

# 3g. memorisation: how many test transactions repeat a (sender, receiver) pair already seen in training?
train_pairs = set(np.unique(sender_code[train_idx] * N_ACCOUNTS + receiver_code[train_idx]).tolist())
test_pair_seen = np.fromiter(((int(a) * N_ACCOUNTS + int(b)) in train_pairs for a, b in zip(sender_code[test_idx], receiver_code[test_idx])), dtype=bool, count=len(test_idx))
print(f"Test rows whose (sender, receiver) pair was seen in training: {test_pair_seen.mean():.1%} overall | "
      f"{test_pair_seen[y[test_idx] == 1].mean():.1%} among laundering rows | {test_pair_seen[y[test_idx] == 0].mean():.1%} among normal rows")
del train_pairs, Xtr, Xte
gc.collect()

[PASS] 3a Train period ends before test period starts: latest train 2022-09-08 16:06:00 | earliest test 2022-09-08 16:06:00
[PASS] 3a No transaction in both train and test: Transaction ID overlap check
[FAIL] 3b Future-invariance of causal features: these features depend on the future: ['Sender Prior Outgoing Amount Total', 'Sender Outgoing Amount Last 24 Hours']
Single-feature ROC-AUC (0.5 = useless, 1.0 / 0.0 = perfect). Top 12:


,Feature,Train AUC,Test AUC,Strength (test)
10,Is New Counterparty Pair,0.721950,0.805962,0.805962
26,Receiver Incoming Txns Last 7 Days,0.463153,0.292134,0.707866
20,Receiver Prior Incoming Txn Count,0.464282,0.300132,0.699868
31,Receiver Bank Prior Txn Count,0.649529,0.644937,0.644937
1,Amount Paid,0.604410,0.639831,0.639831
0,Amount Received,0.604176,0.639657,0.639657
17,Sender Outgoing Txns Last 7 Days,0.497359,0.368195,0.631805
21,Receiver Prior Incoming Amount Total,0.454486,0.375643,0.624357
30,Sender Bank Prior Txn Count,0.636520,0.617676,0.617676
13,Sender Prior Outgoing Amount Mean,0.637192,0.612699,0.612699


[PASS] 3c No single feature is near-perfect: strongest single feature: Is New Counterparty Pair (AUC strength 0.806)


,Model,ROC-AUC,PR-AUC
0,RF - real labels,0.962944,0.050411
1,RF - SHUFFLED labels,0.330709,0.000447
2,No-skill reference,0.500000,0.000668


[PASS] 3d Shuffled-label model is at chance level: ROC-AUC 0.331, PR-AUC 0.00045 (prevalence 0.00067)
Features whose distribution differs most between training and test (KS statistic, 0 = same, 1 = disjoint):


,Feature,KS statistic,Median train,Median test
20,Receiver Prior Incoming Txn Count,0.613600,7.000000,23.000000
8,Day of Week,0.525060,3.000000,4.000000
26,Receiver Incoming Txns Last 7 Days,0.449540,7.000000,16.000000
31,Receiver Bank Prior Txn Count,0.437380,"1,509.000000","4,471.000000"
11,Sender Prior Outgoing Txn Count,0.409560,11.000000,39.000000
17,Sender Outgoing Txns Last 7 Days,0.314400,11.000000,29.000000
30,Sender Bank Prior Txn Count,0.298620,"2,477.500000","7,886.000000"
23,Receiver Prior Unique Senders,0.265000,3.000000,3.000000
10,Is New Counterparty Pair,0.224160,0.000000,0.000000
21,Receiver Prior Incoming Amount Total,0.185760,"74,704.671875","220,286.812500"


[WARN] 3e Train/test feature drift: 6 feature(s) with KS > 0.30, e.g. ['Receiver Prior Incoming Txn Count', 'Day of Week', 'Receiver Incoming Txns Last 7 Days', 'Receiver Bank Prior Txn Count'] (cumulative counters grow with time)


,Feature set,ROC-AUC,PR-AUC
0,all features,0.962944,0.050411
1,without 6 drifting features,0.962142,0.049390


If PR-AUC is similar or better WITHOUT them, remove / rate-normalise those features in notebooks 11-14.
Test rows whose (sender, receiver) pair was seen in training: 96.1% overall | 28.8% among laundering rows | 96.2% among normal rows


116

In [17]:
# =====================================================================================
# CHECK 4 - "XGBoost found 98, Random Forest found 3": leak, or just a different score scale?
# Reads the prediction files written by notebooks 11 / 12 / 14 and compares them on the SAME test rows.
# =====================================================================================
files = {"Random Forest": OUTPUTS_DIR / "random_forest_predictions.csv", "XGBoost": OUTPUTS_DIR / "xgboost_predictions.csv",
         "Logistic Regression": OUTPUTS_DIR / "logistic_regression_predictions.csv", "LSTM": OUTPUTS_DIR / "lstm_predictions.csv"}
preds = {k: pd.read_csv(v, usecols=["Transaction ID", "Actual Is Laundering", "AML Risk Score"]).set_index("Transaction ID") for k, v in files.items() if v.exists()}
if len(preds) < 2:
    print("Run at least two of the model notebooks first (their prediction CSVs are read from outputs/).")
else:
    common_ids = None
    for d in preds.values():
        common_ids = d.index if common_ids is None else common_ids.intersection(d.index)
    print(f"Models found: {list(preds)} | common test rows: {len(common_ids):,}")
    ytrue = next(iter(preds.values())).loc[common_ids, "Actual Is Laundering"].to_numpy()
    id_pos = pd.Series(np.arange(len(tx_ids)), index=tx_ids)
    days = pd.to_datetime(day_num[id_pos.loc[common_ids].to_numpy()] * 86400, unit="s").date
    rows, rank_rows = [], []
    for name, d in preds.items():
        s = d.loc[common_ids, "AML Risk Score"].to_numpy()
        k = max(int(0.01 * len(s)), 1)
        top = np.argsort(-s)[:k]
        rank_rows.append({"Model": name, "Max score": s.max(), "Median score of laundering rows": np.median(s[ytrue == 1]),
                          "ROC-AUC": roc_auc_score(ytrue, s), "PR-AUC": average_precision_score(ytrue, s),
                          "Recall in top 1% of scores": ytrue[top].sum() / ytrue.sum()})
        for thr in (0.5, 0.3, 0.1):
            pr = s >= thr
            rows.append({"Model": name, "Threshold": thr, "TP": int((pr & (ytrue == 1)).sum()), "FP": int((pr & (ytrue == 0)).sum()),
                         "FN": int((~pr & (ytrue == 1)).sum()), "Flagged": int(pr.sum())})
    print("\nThreshold-FREE comparison (ranking quality). Similar PR-AUC / top-1% recall but very different TP at 0.5 => a SCORE-SCALE difference, not a leak:")
    display(pd.DataFrame(rank_rows))
    print("Threshold-dependent comparison:")
    display(pd.DataFrame(rows))
    print("Where do the true positives come from (per day)?")
    tp_day = pd.DataFrame({"Date": days, "Actual": ytrue}).groupby("Date")["Actual"].agg(Rows="size", Laundering="sum")
    for name, d in preds.items():
        s = d.loc[common_ids, "AML Risk Score"].to_numpy()
        tp_day[f"{name} TP@0.5"] = pd.Series(((s >= 0.5) & (ytrue == 1)).astype(int), index=days).groupby(level=0).sum()
    display(tp_day)
    print("Reading guide: a leak makes a model look TOO GOOD (e.g. near-perfect AUC, one feature dominating). A model that catches only a handful at 0.5 but has")
    print("a decent PR-AUC just outputs lower scores (class-weighted RF probabilities rarely reach 0.5) -> pick its threshold on validation (see threshold sweep).")

Models found: ['Random Forest', 'XGBoost', 'Logistic Regression', 'LSTM'] | common test rows: 1,384,810

Threshold-FREE comparison (ranking quality). Similar PR-AUC / top-1% recall but very different TP at 0.5 => a SCORE-SCALE difference, not a leak:


,Model,Max score,Median score of laundering rows,ROC-AUC,PR-AUC,Recall in top 1% of scores
0,Random Forest,0.988437,0.774797,0.973324,0.084896,0.642162
1,XGBoost,0.989465,0.962502,0.977714,0.237869,0.678919
2,Logistic Regression,0.999997,0.983250,0.967325,0.023262,0.334054
3,LSTM,0.998731,0.950963,0.967009,0.046889,0.401081


Threshold-dependent comparison:


,Model,Threshold,TP,FP,FN,Flagged
0,Random Forest,0.500000,658,20109,267,20767
1,Random Forest,0.300000,737,64981,188,65718
2,Random Forest,0.100000,921,287206,4,288127
3,XGBoost,0.500000,879,152014,46,152893
4,XGBoost,0.300000,907,177985,18,178892
5,XGBoost,0.100000,919,211371,6,212290
6,Logistic Regression,0.500000,891,193892,34,194783
7,Logistic Regression,0.300000,922,248563,3,249485
8,Logistic Regression,0.100000,925,329830,0,330755
9,LSTM,0.500000,895,194166,30,195061


Where do the true positives come from (per day)?


,Rows,Laundering,Random Forest TP@0.5,XGBoost TP@0.5,Logistic Regression TP@0.5,LSTM TP@0.5
Date,,,,,,
2022-09-08,210137,123,77,116,116,118
2022-09-09,891573,367,226,343,346,351
2022-09-10,282877,301,229,287,296,292
2022-09-11,77,47,43,47,46,47
2022-09-12,48,27,25,26,27,27
2022-09-13,48,29,27,29,29,29
2022-09-14,31,18,18,18,18,18
2022-09-15,9,6,6,6,6,6
2022-09-16,7,5,5,5,5,5


Reading guide: a leak makes a model look TOO GOOD (e.g. near-perfect AUC, one feature dominating). A model that catches only a handful at 0.5 but has
a decent PR-AUC just outputs lower scores (class-weighted RF probabilities rarely reach 0.5) -> pick its threshold on validation (see threshold sweep).


In [18]:
print("=" * 100); print("CHECKLIST"); print("=" * 100)
for status, name, detail in verdicts:
    print(f"[{status:4}] {name}\n        {detail}")
n_fail = sum(1 for v in verdicts if v[0] == "FAIL"); n_warn = sum(1 for v in verdicts if v[0] == "WARN")
print(f"\n{n_fail} FAIL | {n_warn} WARN | {len(verdicts) - n_fail - n_warn} PASS")
pd.DataFrame(verdicts, columns=["Status", "Check", "Detail"]).to_csv(RESULTS_DIR / "data_quality_and_leakage_checklist.csv", index=False)

CHECKLIST
[PASS] Row / positive counts in expected range
        6,924,049 rows, 3,565 positives
[PASS] Row count equals raw
        raw 6,924,049 vs processed 6,924,049
[PASS] Target column identical to raw
        100.0000% identical
[PASS] Timestamps identical to raw
        100.0000% identical
[WARN] Thin-tail days present
        223 rows / 134 laundering cases - evaluate main window and tail separately
[PASS] 3a Train period ends before test period starts
        latest train 2022-09-08 16:06:00 | earliest test 2022-09-08 16:06:00
[PASS] 3a No transaction in both train and test
        Transaction ID overlap check
[FAIL] 3b Future-invariance of causal features
        these features depend on the future: ['Sender Prior Outgoing Amount Total', 'Sender Outgoing Amount Last 24 Hours']
[PASS] 3c No single feature is near-perfect
        strongest single feature: Is New Counterparty Pair (AUC strength 0.806)
[PASS] 3d Shuffled-label model is at chance level
        ROC-AUC 0.331, PR-A